In [ ]:
!wget -q https://www.cs.ucsb.edu/~william/data/liar_dataset.zip -O liar_dataset.zip
!unzip -q -o liar_dataset.zip

import pandas as pd, os, collections

colnames = [
    "id", "label", "statement", "subject", "speaker", "job_title",
    "state_info", "party_affiliation",
    "barely_true_counts", "false_counts", "half_true_counts",
    "mostly_true_counts", "pants_on_fire_counts",
    "context"
]

train_df = pd.read_csv("train.tsv", sep="\t", header=None, names=colnames, quoting=3)
valid_df = pd.read_csv("valid.tsv", sep="\t", header=None, names=colnames, quoting=3)
test_df  = pd.read_csv("test.tsv",  sep="\t", header=None, names=colnames, quoting=3)

false_set = {"pants-fire", "false", "barely-true"}
true_set  = {"half-true", "mostly-true", "true"}

def to_binary(df):
    df["label_norm"] = df["label"].astype(str).str.strip().str.lower()
    df["labels"] = df["label_norm"].apply(lambda s: 1 if s in true_set else 0)  # 1=true, 0=false
    ### 只保留 statement +labels
    return df[["statement","labels"]]

train_df_b = to_binary(train_df)
valid_df_b = to_binary(valid_df)
test_df_b  = to_binary(test_df)

def stats(df, name):
    cnt = dict(collections.Counter(df["labels"].astype(int).tolist()))
    print(f"{name:5s} | n={len(df):4d} | class balance (0/1) = {cnt}")

stats(train_df_b, "train")
stats(valid_df_b, "valid")
stats(test_df_b,  "test")

# 看 3 条样例（列名：statement / labels）
train_df_b.head(3)

train | n=10269 | class balance (0/1) = {0: 4497, 1: 5772}
valid | n=1284 | class balance (0/1) = {0: 616, 1: 668}
test  | n=1283 | class balance (0/1) = {1: 727, 0: 556}


,statement,labels
0,Says the Annies List political group supports ...,0
1,When did the decline of coal start? It started...,1
2,"Hillary Clinton agrees with John McCain ""by vo...",1


In [ ]:
import time
from sklearn.metrics import f1_score, accuracy_score

def metric_MacroF1_calc(true_labels_list, predicted_labels_list):
    """
    calc Macro-F1, main metric：
    """
    return f1_score(true_labels_list, predicted_labels_list, average="macro")

def metric_Accuracy_calc(true_labels_list, predicted_labels_list):
    """
    calc Accuracy, secondary metrics
    """
    return accuracy_score(true_labels_list, predicted_labels_list)

def metric_latency_calc(text2text_generator_pipeline, input_texts, n_samples_for_timing=300, max_new_tokens=4):
    """
    return mean each_sample_time(ms), float
    """
    total_to_measure = min(n_samples_for_timing, len(input_texts))

    # 热身：第一次生成常常包含编译/缓存，慢且不稳定，不计入统计
    warmup_output = text2text_generator_pipeline(input_texts[0], max_new_tokens=max_new_tokens)

    start_time = time.time()
    for i in range(total_to_measure):
        generated_output = text2text_generator_pipeline(input_texts[i], max_new_tokens=max_new_tokens)
        # 这里只是计时，不需要用 generated_output 的内容
    avg_ms_per_sample = (time.time() - start_time) / total_to_measure * 1000.0
    return avg_ms_per_sample

'''！！！！！！！！！！！！！！！！！'''
'''要注意万一生成式ai生成模型可能偶尔吐出别的词（比如 “barely-true”，怎么办？？
暂且来看，只会是true/false对吧？到时候真生成了乱七八糟的我再想办法,再加限制就行。
Limit max_new_token, enhance paraphrase ability'''

def map_generated_label_text_to_binary_label(generated_label):
    text_lower = str(generated_label).strip().lower()
    return 1 if ("true" in text_lower) else 0

In [ ]:
    ''' The universal config for every model, these are the main config
        the others we use defualt
    '''
Config_Universal = dict(
    max_input_length=256,
    max_new_tokens=64,   # if == 4, only generate "True"/"False"
    random_seed=42,
    train_epochs=3,
    batch_size_train=8,
    batch_size_eval=16,
    gradient_accumulation_steps=4,
    sampling_on = False
)

def get_generation_kwargs_from_config(config_dict):
  sampling_on = config_dict["sampling_on"]
  if sampling_on:
        return dict( do_sample = True,
          temperature = 1.0, top_k = 50, top_p = 0.9,
          max_new_tokens = config_dict["max_new_tokens"],
          truncation = True,
          max_length = config_dict["max_input_length"],
          num_beams=1 )


  return dict( do_sample =False,
    max_new_tokens = config_dict["max_new_tokens"],
    truncation=True,
    max_length = config_dict["max_input_length"],
    num_beams=1 )

In [ ]:

def prompt_setting(statement_text, setting_name):
    """
    - baseline: only statement given
    - instruction only: instruction + statement（Let model know its task）, make output True/False
    - CoT: instruction + CoT + statement. Short Rational first, then label output.
    """
    statement_text = str(statement_text)
    setting_name= str(setting_name).strip()

    # baseline: 只喂原句
    if setting_name == "baseline":
        return statement_text

    # instruction only: instruction + statement（Let model know its task）
    if setting_name == "instruction":
      return ("Instruction: Decide whether the following claim is true or false.\n"
          f'Statement: "{statement_text}"\n'
          "Options: True or False.\n"
          "Output the label only.")

    # instruction + CoT: instruction + CoT + statement（Let model know its task）
    if setting_name == "CoT":
        return ("Instruction: Decide whether the following claim is true or false.\n"
            "Before the final decision, briefly explain your reasoning in 1-2 sentences.\n"
            f'Statement: \"{statement_text}\"\n'
            "Options: True or False.\n"
            "Format:\n"
            "Reasoning: <short explanation>\n"
            "Final label: <True or False>.")

    raise ValueError("setting_name must be 'baseline', 'instruction', or 'CoT'.")


def build_TF_from_label(binary_label_int):
    # In label we used 1=true, 0=false but for model we use "True"/"False"
    return "True" if int(binary_label_int) == 1 else "False"


In [ ]:


##### Prompt Test #############
test_case = test_df_b["statement"].iloc[2]
test_settings = ["baseline", "instruction", "CoT", "WHAT???"]
print( f"Statement: {test_case}\n")
print()
for setting_name in test_settings:
    print(f"--- prompt '{setting_name}' ---")
    try:
        prompt_text = prompt_setting(test_case, setting_name)
        print(prompt_text)
    except ValueError as err:
        # demonstration of illegal input (passing wrong prompt setting )
        print(f"[Error]: {err}")
    print()
print(test_df_b)


Statement: Says John McCain has done nothing to help the vets.


--- prompt 'baseline' ---
Says John McCain has done nothing to help the vets.

--- prompt 'instruction' ---
Instruction: Decide whether the following claim is true or false.
Statement: "Says John McCain has done nothing to help the vets."
Options: True or False.
Output the label only.

--- prompt 'CoT' ---
Instruction: Decide whether the following claim is true or false.
Before the final decision, briefly explain your reasoning in 1-2 sentences.
Statement: "Says John McCain has done nothing to help the vets."
Options: True or False.
Format:
Reasoning: <short explanation>
Final label: <True or False>.

--- prompt 'WHAT???' ---
[Error]: setting_name must be 'baseline', 'instruction', or 'CoT'.

                                              statement  labels
0     Building a wall on the U.S.-Mexico border will...       1
1     Wisconsin is on pace to double the number of l...       0
2     Says John McCain has done nothing t

In [ ]:
# select model
MODEL_NAME = "google/flan-t5-small"
#MODEL_NAME = "facebook/bart-base"

import numpy as np, torch, random
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

'''
# ==== fix random seed====
def set_global_seed(seed_value: int):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)

set_global_seed(Config_Universal["random_seed"])
'''

#======== 加载 tokenizer / model / pipeline ====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)



# ==== Gnerate Key arguments for model from Config_Universal====
gen_kwargs = get_generation_kwargs_from_config(Config_Universal)

print("Device:", device)
print("Loaded:", MODEL_NAME)
print("Generation kwargs:", gen_kwargs)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device: cuda
Loaded: google/flan-t5-small
Generation kwargs: {'do_sample': False, 'max_new_tokens': 64, 'truncation': True, 'max_length': 256, 'num_beams': 1}


In [ ]:
#=============进行烟雾测试（try some prompts)============


# 准备少量 prompts，前三个
instruction_prompts = []
cot_prompts = []

for s in test_df_b["statement"].head(3):
    instruction_prompts.append(prompt_setting(s, "instruction"))
    cot_prompts.append(prompt_setting(s, "CoT"))

# control the process of converting a string into a tensor (how much to truncate, whether to padding, and what tensor format to return).
tokenizer_kwargs = dict(
    padding = False,
    truncation = True,
    max_length = Config_Universal["max_input_length"],
    return_tensors = "pt",
)

# control how the model output is generated (greedy/sampling/beam, how many new tokens to generate)
generate_kwargs = dict(
    do_sample=False,    #greedy；sampling时改为 True 并加 top_p/top_k/temperature
    num_beams=1,
    max_new_tokens=Config_Universal["max_new_tokens"],
)

def generate_outputs_for_prompts(prompts_list):
    outputs_text = []
    for prompt_text in prompts_list:
        print(f"{prompt_text}\n")
        encoded = tokenizer(prompt_text, **tokenizer_kwargs).to(device)

        print(f"ENCODED:\n{encoded}\n")


        with torch.no_grad():
            generated_ids = model.generate(**encoded, **generate_kwargs)
        decoded_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        outputs_text.append(decoded_text)
    return outputs_text

instruction_outputs = generate_outputs_for_prompts(instruction_prompts)
cot_outputs = generate_outputs_for_prompts(cot_prompts)

# 映射并打印
instruction_preds = []
for t in instruction_outputs:
    instruction_preds.append(map_generated_label_text_to_binary_label(t))

cot_preds = []
for t in cot_outputs:
    cot_preds.append(map_generated_label_text_to_binary_label(t))

for i in range(3):
    print(f"\n=== Instruction Example #{i} ===")
    print("PROMPT:\n", instruction_prompts[i])
    print("OUTPUT:\n", instruction_outputs[i])
    print("MAPPED LABEL:", instruction_preds[i])

for i in range(3):
    print(f"\n=== CoT Example #{i} ===")
    print("PROMPT:\n", cot_prompts[i])
    print("OUTPUT:\n", cot_outputs[i])
    print("MAPPED LABEL:", cot_preds[i])


Instruction: Decide whether the following claim is true or false.
Statement: "Building a wall on the U.S.-Mexico border will take literally years."
Options: True or False.
Output the label only.

ENCODED:
{'input_ids': tensor([[21035,    10, 14600,   221,   823,     8,   826,  1988,    19,  1176,
            42,  6136,     5, 16836,    10,    96, 24752,    53,     3,     9,
          1481,    30,     8,   412,     5,   134,     5,    18,   329,   994,
          5807,  4947,    56,   240,  6672,   203,   535, 17011,    10, 10998,
            42, 10747,     7,    15,     5,  3387,  2562,     8,  3783,   163,
             5,     1]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1]], device='cuda:0')}

Instruction: Decide whether the following claim is true or false.
Statement: "Wisconsin is on pace to double the number o

In [ ]:
#MODEL_NAME = "google/flan-t5-small"
MODEL_NAME = "facebook/bart-base"

import random, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 固定随机性（和 Config_Universal 保持一致）
def set_global_seed(seed_value: int):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)

set_global_seed(Config_Universal["random_seed"])

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Loaded:", MODEL_NAME, "| Device:", device)

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loaded: facebook/bart-base | Device: cuda


In [ ]:
# 统一：输入用 "instruction" 模板；目标只生成 "True"/"False"
def build_input_text(statement_text: str) -> str:
    return (
        "Instruction: Decide whether the following claim is true or false.\n"
        f'Statement: "{statement_text}"\n'
        "Options: True or False.\n"
        "Output the label only."
    )

def build_target_text_from_label_int(binary_label_int: int) -> str:
    return "True" if int(binary_label_int) == 1 else "False"

# 把一个 DataFrame（带列 statement / labels）转换为 Hugging Face Dataset 所需的 dict 列表
def dataframe_to_examples(df):
    inputs  = []
    targets = []
    for s, y in zip(df["statement"].tolist(), df["labels"].astype(int).tolist()):
        inputs.append(build_input_text(str(s)))
        targets.append(build_target_text_from_label_int(y))
    return inputs, targets

train_inputs, train_targets = dataframe_to_examples(train_df_b)
valid_inputs, valid_targets = dataframe_to_examples(valid_df_b)
test_inputs,  test_targets  = dataframe_to_examples(test_df_b)

print("train/valid/test sizes:", len(train_inputs), len(valid_inputs), len(test_inputs))
print("Sample INPUT:\n", train_inputs[0])
print("Sample TARGET:", train_targets[0])


train/valid/test sizes: 10269 1284 1283
Sample INPUT:
 Instruction: Decide whether the following claim is true or false.
Statement: "Says the Annies List political group supports third-trimester abortions on demand."
Options: True or False.
Output the label only.
Sample TARGET: False


In [ ]:
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq

max_input_length  = Config_Universal["max_input_length"]
max_target_length = 4                      # "True"/"False" 最多 1–2 个 token，4 足够

# 先构造 Hugging Face Dataset（更方便 Trainer 使用）
train_ds = Dataset.from_dict({"input_text": train_inputs, "target_text": train_targets})
valid_ds = Dataset.from_dict({"input_text": valid_inputs, "target_text": valid_targets})
test_ds  = Dataset.from_dict({"input_text":  test_inputs, "target_text":  test_targets})

def tokenize_example(example):
    # 编码输入
    model_inputs = tokenizer(
        example["input_text"],
        truncation=True,
        max_length=max_input_length
    )
    # 编码目标（用 text_target，而不是 as_target_tokenizer）
    labels = tokenizer(
        text_target=example["target_text"],
        truncation=True,
        max_length=max_target_length
    )["input_ids"]
    model_inputs["labels"] = labels
    return model_inputs
train_tokenized = train_ds.map(tokenize_example, remove_columns=train_ds.column_names)
valid_tokenized = valid_ds.map(tokenize_example, remove_columns=valid_ds.column_names)
test_tokenized  = test_ds.map( tokenize_example, remove_columns=test_ds.column_names)

# 动态 padding 的 collator（自动对齐 batch 内不同长度）
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

print(train_tokenized[0])


Map:   0%|          | 0/10269 [00:00<?, ? examples/s]

Map:   0%|          | 0/1284 [00:00<?, ? examples/s]

Map:   0%|          | 0/1283 [00:00<?, ? examples/s]

{'input_ids': [0, 23754, 31470, 35, 1502, 1949, 549, 5, 511, 2026, 16, 1528, 50, 3950, 4, 50118, 46791, 35, 22, 104, 4113, 5, 3921, 918, 9527, 559, 333, 4548, 371, 12, 4328, 38417, 17600, 15, 1077, 72, 50118, 47261, 35, 7447, 50, 35297, 4, 50118, 48293, 5, 6929, 129, 4, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [0, 46659, 2]}


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import transformers
print("Transformers version:", transformers.__version__)

# --- 指标函数：解码 → "True"/"False" → 0/1 ---
def compute_metrics_for_generation(eval_pred):
    pred_ids = eval_pred.predictions
    label_ids = eval_pred.label_ids

    # 一些版本里 predictions 是 tuple(sequences, …)
    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]

    # 解码
    pred_texts = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)

    # labels 里 -100 需要换成 pad 再解码
    import numpy as np
    label_ids = np.array(label_ids)
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    label_texts = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    def text_to_int(t: str) -> int:
        return 1 if "true" in str(t).strip().lower() else 0

    y_true = [text_to_int(t) for t in label_texts]
    y_pred = [text_to_int(t) for t in pred_texts]

    macro_f1 = metric_MacroF1_calc(y_true, y_pred)
    acc      = metric_Accuracy_calc(y_true, y_pred)
    return {"macro_f1": macro_f1, "accuracy": acc}

# --- 先用新接口 eval_strategy；若失败再回退 evaluation_strategy ---
common_kwargs = dict(
    output_dir="runs_t5_instruction_sft",
    num_train_epochs=Config_Universal["train_epochs"],
    per_device_train_batch_size=Config_Universal["batch_size_train"],
    per_device_eval_batch_size=Config_Universal["batch_size_eval"],
    gradient_accumulation_steps=Config_Universal["gradient_accumulation_steps"],
    predict_with_generate=True,                               # 评估时用 generate()
    generation_max_length=Config_Universal["max_new_tokens"], # 与推理一致
    save_strategy="no",
    logging_steps=50,
    report_to=[],
    seed=Config_Universal["random_seed"],
)

try:
    training_args = Seq2SeqTrainingArguments(
        eval_strategy="epoch",   # 新版参数名
        **common_kwargs
    )
except TypeError:
    training_args = Seq2SeqTrainingArguments(
        evaluation_strategy="epoch",  # 旧版参数名
        **common_kwargs
    ),

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    data_collator=data_collator,
    train_dataset=train_tokenized,
    eval_dataset=valid_tokenized,
    compute_metrics=compute_metrics_for_generation
)

train_result = trainer.train()
print("Training done.")

# Dev 集评估（不用再传 max_length；用上面的 generation_max_length）
dev_metrics = trainer.evaluate()
print("Dev metrics:", dev_metrics)


Transformers version: 4.57.2


/tmp/ipython-input-1549144404.py:59: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.170900,0.238667,0.643603,0.644081
2,0.123600,0.264332,0.628417,0.629283
3,0.116000,0.318643,0.601975,0.612928


Training done.


Dev metrics: {'eval_loss': 0.31864285469055176, 'eval_macro_f1': 0.6019751922144944, 'eval_accuracy': 0.6129283489096573, 'eval_runtime': 12.9494, 'eval_samples_per_second': 99.155, 'eval_steps_per_second': 6.255, 'epoch': 3.0}


In [ ]:
# 用同样的生成上限评测 test（统一公平）
test_predictions = trainer.predict(test_tokenized, max_length=max_input_length)
# decode + 映射（和 compute_metrics 里同样的逻辑）
pred_texts  = tokenizer.batch_decode(test_predictions.predictions, skip_special_tokens=True)

# 把 label -100 → pad 再解码
label_ids = test_predictions.label_ids
label_ids[label_ids == -100] = tokenizer.pad_token_id
label_texts = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

def label_text_to_int(t: str) -> int:
    return 1 if "true" in str(t).strip().lower() else 0

y_true = [label_text_to_int(t) for t in label_texts]
y_pred = [label_text_to_int(t) for t in pred_texts]

test_macro_f1 = metric_MacroF1_calc(y_true, y_pred)
test_acc      = metric_Accuracy_calc(y_true, y_pred)
print(f"[Instruction SFT @ TEST] Macro-F1={test_macro_f1:.4f} | Accuracy={test_acc:.4f}")

# （可选）测一下 test 推理延迟（与你的 latency 函数保持一致）
from time import time
def run_latency(prompts, n=300):
    # 构造 prompts（instruction 模板）
    sample_prompts = [build_input_text(s) for s in test_df_b["statement"].head(n)]
    # 直接用你之前的 metric_latency_calc 也行；这里用最朴素的测法
    import time
    start = time.time()
    for p in sample_prompts:
        enc = tokenizer(p, truncation=True, max_length=max_input_length, return_tensors="pt").to(device)
        with torch.no_grad():
            _ = model.generate(**enc, do_sample=False, num_beams=1, max_new_tokens=Config_Universal["max_new_tokens"])
    avg_ms = (time.time() - start) * 1000.0 / len(sample_prompts)
    return avg_ms

avg_ms = run_latency(test_df_b, n=100)  # 先测100条，正式报告再设为300
print(f"Avg latency (100 samples): {avg_ms:.1f} ms/sample")


[Instruction SFT @ TEST] Macro-F1=0.5780 | Accuracy=0.6041
Avg latency (100 samples): 27.2 ms/sample


In [ ]:
# 测 300 条的平均/ P95 延迟（instruction 模板；与主实验一致：greedy, max_new_tokens=64）
import time, numpy as np

def measure_latency_ms_list(prompts_list, batch_count=300):
    times_ms = []
    for p in prompts_list[:batch_count]:
        start = time.time()
        enc = tokenizer(
            p,
            truncation=True,
            max_length=Config_Universal["max_input_length"],
            return_tensors="pt"
        ).to(model.device)
        with torch.no_grad():
            _ = model.generate(
                **enc,
                do_sample=False,
                num_beams=1,
                max_new_tokens=Config_Universal["max_new_tokens"]
            )
        times_ms.append((time.time() - start) * 1000.0)
    return times_ms

# 构造 300 条 instruction prompts
prompts_300 = [
    "Instruction: Decide whether the following claim is true or false.\n"
    f'Statement: "{s}"\n'
    "Options: True or False.\n"
    "Output the label only."
    for s in test_df_b["statement"].head(300)
]

latency_list = measure_latency_ms_list(prompts_300, batch_count=300)
avg_ms = float(np.mean(latency_list))
p95_ms = float(np.percentile(latency_list, 95))
print(f"[Instruction SFT] Latency over 300 samples: avg={avg_ms:.1f} ms | p95={p95_ms:.1f} ms")


[Instruction SFT] Latency over 300 samples: avg=29.6 ms | p95=40.5 ms


In [ ]:
from torch.utils.data import DataLoader

def average_generated_tokens_v2(tokenized_dataset, batch_size=32):
    loader = DataLoader(
        tokenized_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=data_collator  # 动态 padding
    )
    gen_kwargs = dict(
        do_sample=False,
        num_beams=1,
        max_new_tokens=Config_Universal["max_new_tokens"]
    )

    total_tokens, total_samples = 0, 0
    model.eval()
    with torch.no_grad():
        for batch in loader:
            inputs = {
                "input_ids": batch["input_ids"].to(model.device),
                "attention_mask": batch["attention_mask"].to(model.device),
            }
            gen_ids = model.generate(**inputs, **gen_kwargs)  # [B, T_out]
            # 先解码为字符串，再用 add_special_tokens=False 重新编码计数
            texts = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
            enc = tokenizer(
                texts,
                add_special_tokens=False,
                return_tensors=None
            )
            # enc["input_ids"] 是 list[list[int]]，每个子列表长度就是“内容 token 数”
            lengths = [len(x) for x in enc["input_ids"]]
            total_tokens += sum(lengths)
            total_samples += len(lengths)

    avg_tokens = total_tokens / max(1, total_samples)
    return avg_tokens

avg_gen_tokens_instruction_bart = average_generated_tokens_v2(
    test_tokenized,
    batch_size=Config_Universal["batch_size_eval"]
)
print(f"[BART Instruction SFT] Avg generated tokens on TEST: {avg_gen_tokens_instruction_bart:.2f}")


[BART Instruction SFT] Avg generated tokens on TEST: 1.00


In [ ]:
# 抽查 BART 在 test 上的输出 token 数分布
gen_kwargs = dict(do_sample=False, num_beams=1, max_new_tokens=Config_Universal["max_new_tokens"])

texts = []
model.eval()
with torch.no_grad():
    for i in range(50):  # 先看前50条
        p = (
            "Instruction: Decide whether the following claim is true or false.\n"
            f'Statement: "{test_df_b["statement"].iloc[i]}"\n'
            "Options: True or False.\n"
            "Output the label only."
        )
        enc = tokenizer(p, truncation=True, max_length=Config_Universal["max_input_length"], return_tensors="pt").to(model.device)
        ids = model.generate(**enc, **gen_kwargs)
        texts.append(tokenizer.decode(ids[0], skip_special_tokens=True))

# 重新编码计数（不加特殊符号），看分布
enc = tokenizer(texts, add_special_tokens=False, return_tensors=None)
lengths = [len(x) for x in enc["input_ids"]]
from collections import Counter
print("length histogram:", Counter(lengths))
print("avg length:", sum(lengths)/len(lengths))
print("sample outputs:", texts[:5])


length histogram: Counter({1: 50})
avg length: 1.0
sample outputs: ['True', 'True', 'False', 'True', 'False']


**开始CoT**

In [ ]:
# === 更快的教师生成（batched + 动态padding + 进度条 + 分批保存） ===
TEACHER_MODEL_NAME = "google/flan-t5-base"

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import torch, json, os, numpy as np, re

teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_NAME)
teacher_model     = AutoModelForSeq2SeqLM.from_pretrained(TEACHER_MODEL_NAME).to(device)
teacher_model.eval()

# 可选：让生成在半精度跑（T4/A100 一般支持；若报错就注释掉）
try:
    teacher_model.half()
except Exception:
    pass

# ---- 规范化函数（与你此前一致）----
RE_FINAL   = re.compile(r'final\s*label\s*:\s*(true|false)\b', re.IGNORECASE)
RE_REASON  = re.compile(r'^reasoning\s*:\s*(.+)$', re.IGNORECASE | re.MULTILINE)

def normalize_teacher_output(raw_text: str, gold_label_text: str) -> str:
    text = str(raw_text).strip()
    m_reason = RE_REASON.search(text)
    if m_reason:
        reasoning = m_reason.group(1).strip()
    else:
        reasoning = ("The claim is consistent with publicly available evidence and reputable sources."
                     if gold_label_text == "True"
                     else "The claim contradicts public records or lacks verifiable supporting evidence.")
    return f"Reasoning: {reasoning}\nFinal label: {gold_label_text}"

def build_teacher_prompt(statement_text: str, gold_label_text: str) -> str:
    return (
        "You are an expert fact-checking assistant.\n"
        "Task: Decide whether the claim is true or false. Provide a brief reasoning (1–2 sentences), "
        "then output the final label EXACTLY as True or False on the last line.\n\n"
        "Example:\n"
        "Claim: \"The earth orbits the sun.\"\n"
        "Gold label: True\n"
        "Reasoning: This is a well-established scientific fact confirmed by astronomy.\n"
        "Final label: True\n\n"
        "Now do the following:\n"
        f"Claim: \"{statement_text}\"\n"
        f"Gold label: {gold_label_text}\n"
        "Output exactly two lines:\n"
        "Reasoning: <short explanation>\n"
        "Final label: <True or False>"
    )

# ---- 小型 Dataset 方便 DataLoader 批量化 ----
class TeacherPromptDataset(Dataset):
    def __init__(self, statements: list[str], gold_labels_text: list[str]):
        self.statements = statements
        self.gold = gold_labels_text
    def __len__(self):
        return len(self.statements)
    def __getitem__(self, idx):
        return build_teacher_prompt(self.statements[idx], self.gold[idx])

def generate_teacher_targets_batched(df_split,
                                     batch_size: int = 16,
                                     max_new_tokens_teacher: int = 128,
                                     save_every_batches: int = 50,
                                     save_path: str | None = None,
                                     preview_count: int = 3):
    statements = df_split["statement"].astype(str).tolist()
    labels_int = df_split["labels"].astype(int).tolist()
    gold_texts = ["True" if x == 1 else "False" for x in labels_int]

    dataset = TeacherPromptDataset(statements, gold_texts)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)

    all_targets = []
    seen = 0

    # 先打印几条预览（单条跑）
    for i in range(min(preview_count, len(dataset))):
        prompt = dataset[i]
        enc = teacher_tokenizer(prompt, truncation=True,
                                max_length=Config_Universal["max_input_length"],
                                return_tensors="pt").to(device)
        with torch.no_grad():
            out_ids = teacher_model.generate(**enc, do_sample=False, num_beams=1,
                                             max_new_tokens=max_new_tokens_teacher)
        raw = teacher_tokenizer.decode(out_ids[0], skip_special_tokens=True)
        norm = normalize_teacher_output(raw, gold_texts[i])
        print(f"\n=== Teacher preview #{i} ===")
        print("PROMPT:\n", prompt)
        print("RAW:\n", raw)
        print("NORMALIZED:\n", norm)

    # 批量生成
    pbar = tqdm(loader, desc="Generating CoT targets (teacher, batched)")
    for batch_prompts in pbar:
        # tokenizer 支持批量字符串
        enc = teacher_tokenizer(
            list(batch_prompts),
            truncation=True,
            max_length=Config_Universal["max_input_length"],
            padding=True,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            out_ids = teacher_model.generate(
                **enc,
                do_sample=False,
                num_beams=1,
                max_new_tokens=max_new_tokens_teacher
            )

        # 批量解码 & 规范化
        batch_texts = teacher_tokenizer.batch_decode(out_ids, skip_special_tokens=True)
        # 对应的 gold 从 seen 开始取
        for j, raw in enumerate(batch_texts):
            gold = gold_texts[seen + j]
            norm = normalize_teacher_output(raw, gold)
            all_targets.append(norm)
        seen += len(batch_texts)

        # 定期落盘（可断点续跑）
        if save_path and (seen // batch_size) % save_every_batches == 0:
            with open(save_path, "w", encoding="utf-8") as f:
                for t in all_targets:
                    f.write(json.dumps({"target": t}) + "\n")
            pbar.set_postfix(saved=len(all_targets))

    # 最后一轮保存
    if save_path:
        with open(save_path, "w", encoding="utf-8") as f:
            for t in all_targets:
                f.write(json.dumps({"target": t}) + "\n")

    return all_targets
'''
# === 先跑 200 条小样（可观察速度 & 格式），再全量 ===
cot_targets_preview = generate_teacher_targets_batched(
    train_df_b.head(200),
    batch_size=16,
    max_new_tokens_teacher=128,
    save_every_batches=10,
    save_path=None,           # 预览阶段不落盘
    preview_count=2
)
print(f"\nPreview generated {len(cot_targets_preview)} CoT targets.")
'''
# === 全量（会显著更快 & 可断点续跑）===
cot_targets_full = generate_teacher_targets_batched(
      train_df_b,
      batch_size=16,
      max_new_tokens_teacher=128,
      save_every_batches=50,
      #save_path="cot_targets_train.jsonl",   # 断点文件（可复用）
      preview_count=0)

print(f"Generated {len(cot_targets_full)} CoT targets for full training set.")


Generating CoT targets (teacher, batched):   0%|          | 0/642 [00:00<?, ?it/s]

Generated 10269 CoT targets for full training set.


In [ ]:
# =========================
# Cell B (updated): 组装 CoT 版数据 + Tokenize
# =========================

import json, os
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq

# ---- 1) 读取教师生成的 CoT 目标：优先用内存变量 cot_targets_full；否则从 jsonl 文件加载 ----
def load_cot_targets_for_train(
    df_train,
    in_memory_targets: list[str] | None = None,
    jsonl_path: str | None = None
) -> list[str]:
    if in_memory_targets is not None:
        cot_targets = list(in_memory_targets)
    elif jsonl_path is not None and os.path.exists(jsonl_path):
        cot_targets = []
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                cot_targets.append(obj["target"])
    else:
        raise ValueError("No CoT targets found. Provide in_memory_targets=cot_targets_full or jsonl_path='cot_targets_train.jsonl'.")

    # 安全校验：数量必须与 train_df_b 行数一致
    if len(cot_targets) != len(df_train):
        raise ValueError(f"Number of CoT targets ({len(cot_targets)}) != train size ({len(df_train)}). Check your source.")
    return cot_targets

# 你可以二选一：
# 1) 如果你已在内存里有 cot_targets_full：
# cot_targets_train = load_cot_targets_for_train(train_df_b, in_memory_targets=cot_targets_full)
# 2) 如果你跑的是批量+落盘版本（例如 'cot_targets_train.jsonl'）：
# cot_targets_train = load_cot_targets_for_train(train_df_b, jsonl_path="cot_targets_train.jsonl")

# === 这里先尝试内存变量；若没有，请改成 jsonl_path 版本 ===
try:
    cot_targets_train = load_cot_targets_for_train(train_df_b, in_memory_targets=cot_targets_full)
except NameError:
    cot_targets_train = load_cot_targets_for_train(train_df_b, jsonl_path="cot_targets_train.jsonl")

# ---- 2) 构造 CoT 版 input/target（与之前定义的模板保持一致）----
def build_cot_input_text(statement_text: str) -> str:
    return (
        "Instruction: Decide whether the following claim is true or false.\n"
        "Before the final decision, briefly explain your reasoning in 1-2 sentences.\n"
        f'Statement: "{statement_text}"\n'
        "Options: True or False.\n"
        "Format:\n"
        "Reasoning: <short explanation>\n"
        "Final label: <True or False>."
    )

def dataframe_to_cot_examples_train(df_split, teacher_targets: list[str]):
    assert len(df_split) == len(teacher_targets)
    inputs  = [build_cot_input_text(str(s)) for s in df_split["statement"].tolist()]
    targets = [str(t) for t in teacher_targets]   # 目标是“Reasoning + Final label: X”
    return inputs, targets

def dataframe_to_cot_examples_eval(df_split):
    # Dev/Test 的 target 仍为 "True"/"False"；评测时生成→解析→算分
    inputs  = [build_cot_input_text(str(s)) for s in df_split["statement"].tolist()]
    targets = ["True" if int(y) == 1 else "False" for y in df_split["labels"].astype(int).tolist()]
    return inputs, targets

cot_train_inputs, cot_train_targets = dataframe_to_cot_examples_train(train_df_b, cot_targets_train)
cot_valid_inputs, cot_valid_targets = dataframe_to_cot_examples_eval(valid_df_b)
cot_test_inputs,  cot_test_targets  = dataframe_to_cot_examples_eval(test_df_b)

print("CoT train/dev/test sizes:", len(cot_train_inputs), len(cot_valid_inputs), len(cot_test_inputs))
print("Sample CoT INPUT:\n", cot_train_inputs[0])
print("Sample CoT TARGET:\n", cot_train_targets[0])

# ---- 3) 构造 HF Dataset 并 Tokenize（注意：target 截断长度提高到 128）----
max_input_length  = Config_Universal["max_input_length"]    # 仍为 256
max_target_length = 128                                     # CoT 目标更长

train_cot_ds = Dataset.from_dict({"input_text": cot_train_inputs, "target_text": cot_train_targets})
valid_cot_ds = Dataset.from_dict({"input_text": cot_valid_inputs, "target_text": cot_valid_targets})
test_cot_ds  = Dataset.from_dict({"input_text": cot_test_inputs,  "target_text": cot_test_targets})

def tokenize_example_cot(example):
    # 输入侧
    model_inputs = tokenizer(
        example["input_text"],
        truncation=True,
        max_length=max_input_length
    )
    # 目标侧：使用 text_target 的现代写法
    labels = tokenizer(
        text_target=example["target_text"],
        truncation=True,
        max_length=max_target_length
    )["input_ids"]
    model_inputs["labels"] = labels
    return model_inputs

train_tokenized_cot = train_cot_ds.map(tokenize_example_cot, remove_columns=train_cot_ds.column_names)
valid_tokenized_cot = valid_cot_ds.map(tokenize_example_cot, remove_columns=valid_cot_ds.column_names)
test_tokenized_cot  = test_cot_ds.map( tokenize_example_cot, remove_columns=test_cot_ds.column_names)

# 动态 padding 的 collator（与 Instruction 相同，但这里单独命名以免混淆）
data_collator_cot = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

print("Tokenized sample (CoT):", train_tokenized_cot[0])


CoT train/dev/test sizes: 10269 1284 1283
Sample CoT INPUT:
 Instruction: Decide whether the following claim is true or false.
Before the final decision, briefly explain your reasoning in 1-2 sentences.
Statement: "Says the Annies List political group supports third-trimester abortions on demand."
Options: True or False.
Format:
Reasoning: <short explanation>
Final label: <True or False>.
Sample CoT TARGET:
 Reasoning: The claim contradicts public records or lacks verifiable supporting evidence.
Final label: False


Map:   0%|          | 0/10269 [00:00<?, ? examples/s]

Map:   0%|          | 0/1284 [00:00<?, ? examples/s]

Map:   0%|          | 0/1283 [00:00<?, ? examples/s]

Tokenized sample (CoT): {'input_ids': [0, 23754, 31470, 35, 1502, 1949, 549, 5, 511, 2026, 16, 1528, 50, 3950, 4, 50118, 17206, 5, 507, 568, 6, 7478, 3922, 110, 20511, 11, 112, 12, 176, 11305, 4, 50118, 46791, 35, 22, 104, 4113, 5, 3921, 918, 9527, 559, 333, 4548, 371, 12, 4328, 38417, 17600, 15, 1077, 72, 50118, 47261, 35, 7447, 50, 35297, 4, 50118, 48587, 35, 50118, 48147, 154, 35, 28696, 20263, 8257, 15698, 50118, 38083, 6929, 35, 28696, 36948, 50, 35297, 48691, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [0, 48147, 154, 35, 20, 2026, 37820, 285, 2189, 50, 14026, 4342, 21584, 3117, 1283, 4, 50118, 38083, 6929, 35, 35297, 2]}


In [ ]:
# === Cell C: Train CoT SFT (T5-small by default) ===
import re, numpy as np, transformers
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

print("Transformers version:", transformers.__version__)

# 解析预测文本中的最终标签：优先抓 "Final label: X"，否则取文中最后一次完整的 True/False
RE_FINAL = re.compile(r'final\s*label\s*:\s*(true|false)\b', re.IGNORECASE)
RE_WORD  = re.compile(r'\b(true|false)\b', re.IGNORECASE)

def parse_label_text_to_int(text: str) -> int:
    s = str(text).strip()
    m = RE_FINAL.search(s)
    if m:
        return 1 if m.group(1).lower() == "true" else 0
    words = RE_WORD.findall(s)
    if words:
        return 1 if words[-1].lower() == "true" else 0
    return 0  # 保守

def compute_metrics_for_generation_cot(eval_pred):
    pred_ids  = eval_pred.predictions
    label_ids = eval_pred.label_ids
    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]
    pred_texts = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)

    label_ids = np.array(label_ids)
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    label_texts = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    y_true = [1 if "true" in t.strip().lower() else 0 for t in label_texts]  # dev/test 的 target 仍是 "True"/"False"
    y_pred = [parse_label_text_to_int(t) for t in pred_texts]

    macro_f1 = metric_MacroF1_calc(y_true, y_pred)
    acc      = metric_Accuracy_calc(y_true, y_pred)
    return {"macro_f1": macro_f1, "accuracy": acc}

common_kwargs = dict(
    output_dir="runs_t5_cot_sft",
    num_train_epochs=Config_Universal["train_epochs"],
    per_device_train_batch_size=Config_Universal["batch_size_train"],
    per_device_eval_batch_size=Config_Universal["batch_size_eval"],
    gradient_accumulation_steps=Config_Universal["gradient_accumulation_steps"],
    predict_with_generate=True,
    generation_max_length=Config_Universal["max_new_tokens"],  # 与 Instruction 一致（公平）
    save_strategy="no",
    logging_steps=50,
    report_to=[],
    seed=Config_Universal["random_seed"],
)

# 兼容新老参数名
try:
    training_args_cot = Seq2SeqTrainingArguments(eval_strategy="epoch", **common_kwargs)
except TypeError:
    training_args_cot = Seq2SeqTrainingArguments(evaluation_strategy="epoch", **common_kwargs)

trainer_cot = Seq2SeqTrainer(
    model=model,
    args=training_args_cot,
    tokenizer=tokenizer,
    data_collator=data_collator_cot,   # 注意：使用 CoT 的 collator
    train_dataset=train_tokenized_cot,
    eval_dataset=valid_tokenized_cot,
    compute_metrics=compute_metrics_for_generation_cot
)

train_result_cot = trainer_cot.train()
print("CoT training done.")
dev_metrics_cot = trainer_cot.evaluate()
print("Dev (CoT) metrics:", dev_metrics_cot)


Transformers version: 4.57.2


/tmp/ipython-input-1245661337.py:59: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer_cot = Seq2SeqTrainer(


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.024000,6.001791,0.634581,0.637072
2,0.019400,6.421785,0.619556,0.625389
3,0.011900,7.311031,0.609597,0.616044


CoT training done.


Dev (CoT) metrics: {'eval_loss': 7.311030864715576, 'eval_macro_f1': 0.6095967080930363, 'eval_accuracy': 0.6160436137071651, 'eval_runtime': 50.0123, 'eval_samples_per_second': 25.674, 'eval_steps_per_second': 1.62, 'epoch': 3.0}


In [ ]:
# === Cell D: Evaluate on TEST (CoT SFT) ===
test_predictions_cot = trainer_cot.predict(test_tokenized_cot)
pred_ids  = test_predictions_cot.predictions[0] if isinstance(test_predictions_cot.predictions, tuple) else test_predictions_cot.predictions
label_ids = test_predictions_cot.label_ids

pred_texts  = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
label_ids[label_ids == -100] = tokenizer.pad_token_id
label_texts = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

y_true = [1 if "true" in t.strip().lower() else 0 for t in label_texts]
y_pred = [parse_label_text_to_int(t) for t in pred_texts]

test_macro_f1_cot = metric_MacroF1_calc(y_true, y_pred)
test_acc_cot      = metric_Accuracy_calc(y_true, y_pred)
print(f"[CoT SFT @ TEST] Macro-F1={test_macro_f1_cot:.4f} | Accuracy={test_acc_cot:.4f}")


[CoT SFT @ TEST] Macro-F1=0.5902 | Accuracy=0.6126


In [ ]:
# === Cell E: Stable latency for CoT (n=300, batched) ===
import time, numpy as np, torch
from torch.utils.data import DataLoader

def measure_latency_stable(prompts, batch_size=8, n_samples=300):
    prompts = list(prompts)[:n_samples]
    enc = tokenizer(
        prompts,
        truncation=True,
        max_length=Config_Universal["max_input_length"],
        padding=True,
        return_tensors="pt"
    )
    dataset = torch.utils.data.TensorDataset(enc["input_ids"], enc["attention_mask"])
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    device = model.device
    inputs_gpu = [{"input_ids": b[0].to(device, non_blocking=True),
                   "attention_mask": b[1].to(device, non_blocking=True)} for b in loader]

    model.eval()
    with torch.no_grad():
        for warm in inputs_gpu[:2]:
            _ = model.generate(**warm, do_sample=False, num_beams=1,
                               max_new_tokens=Config_Universal["max_new_tokens"])

    batch_times_ms, per_sample_ms = [], []
    with torch.no_grad():
        for batch_inputs in inputs_gpu:
            torch.cuda.synchronize(device)
            t0 = time.perf_counter()
            _ = model.generate(**batch_inputs, do_sample=False, num_beams=1,
                               max_new_tokens=Config_Universal["max_new_tokens"])
            torch.cuda.synchronize(device)
            t1 = time.perf_counter()
            batch_times_ms.append((t1 - t0) * 1000.0)
            bsz = batch_inputs["input_ids"].size(0)
            per_sample_ms.extend([batch_times_ms[-1] / bsz] * bsz)

    avg_ms = float(np.mean(per_sample_ms))
    p95_ms = float(np.percentile(per_sample_ms, 95))
    return avg_ms, p95_ms

# 构造 300 条 CoT prompts（与你训练/评测的 CoT 模板一致）
prompts_cot_300 = [
    "Instruction: Decide whether the following claim is true or false.\n"
    "Before the final decision, briefly explain your reasoning in 1-2 sentences.\n"
    f'Statement: "{s}"\n'
    "Options: True or False.\n"
    "Format:\n"
    "Reasoning: <short explanation>\n"
    "Final label: <True or False>."
    for s in test_df_b["statement"].head(300)
]

avg_ms_cot, p95_ms_cot = measure_latency_stable(prompts_cot_300, batch_size=8, n_samples=300)
print(f"[CoT SFT · stable] avg={avg_ms_cot:.1f} ms | p95={p95_ms_cot:.1f} ms (n=300)")


[CoT SFT · stable] avg=28.7 ms | p95=36.6 ms (n=300)


In [ ]:
# === Cell F: Avg generated tokens on TEST (CoT) ===
from torch.utils.data import DataLoader

def average_generated_tokens_v2(tokenized_dataset, batch_size=32):
    loader = DataLoader(tokenized_dataset, batch_size=batch_size, shuffle=False, collate_fn=data_collator_cot)
    gen_kwargs = dict(do_sample=False, num_beams=1, max_new_tokens=Config_Universal["max_new_tokens"])
    total_tokens, total_samples = 0, 0
    model.eval()
    with torch.no_grad():
        for batch in loader:
            inputs = {
                "input_ids": batch["input_ids"].to(model.device),
                "attention_mask": batch["attention_mask"].to(model.device),
            }
            gen_ids = model.generate(**inputs, **gen_kwargs)
            texts = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
            enc = tokenizer(texts, add_special_tokens=False, return_tensors=None)
            lengths = [len(x) for x in enc["input_ids"]]
            total_tokens += sum(lengths)
            total_samples += len(lengths)
    return total_tokens / max(1, total_samples)

avg_gen_tokens_cot = average_generated_tokens_v2(test_tokenized_cot, batch_size=Config_Universal["batch_size_eval"])
print(f"[CoT SFT] Avg generated tokens on TEST: {avg_gen_tokens_cot:.2f}")


[CoT SFT] Avg generated tokens on TEST: 20.00


In [ ]:
print("Current model:", getattr(model, "name_or_path", None))


Current model: facebook/bart-base
